# Elevator fleet — the simulator

Produces the fleet of the lecture-10 module onto two Kafka topics.

**Run the early part of this notebook first**, then alternate with
`2_fleet_monitoring.ipynb`: it will tell you when to come back and run a section.

The trace is **designed, not random**. The background traffic exists so that the
fleet aggregation and the sizing arithmetic have something plausible to work on;
every other section isolates exactly one thing. That is the same discipline the EPL
modules use, and the reason the results are worth reading.

### The client is already installed

There is no `pip` cell in this module. `docker-compose.yml` mounts a small script
into the notebook image's `before-notebook.d/`, so `confluent-kafka` is installed
once when the container starts — in the container's own environment, before the
server is up.

That is not a stylistic preference. An install from a cell has to launch a
subprocess from the kernel, and that is exactly what breaks when the image is being
emulated rather than run natively: the kernel wedges, unkillably, and the cell shows
no output and ignores the interrupt. Better to have nothing in the lecture depend on
it. The next cell is therefore a **check**, not a wait — and if it fails, the install
from a terminal on your host is one line:

```
docker exec -it fleet-notebook pip install confluent-kafka==2.5.0
```

In [1]:
import confluent_kafka
print("confluent-kafka", confluent_kafka.version()[0])

confluent-kafka 2.5.0


In [2]:
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka import SerializingProducer
from confluent_kafka.serialization import StringSerializer
import json, time

bootstrap_server = "kafka:9092"
TOPIC_EVENTS = "elevator-events"
TOPIC_FAULTS = "elevator-faults"

admin = AdminClient({"bootstrap.servers": bootstrap_server})

## The two topics, and why they are two

Door movements are **frequent and low value per event**; faults are **rare and high
value**. Two topics, so that they can have different retention, different partition
counts and different consumer SLAs — which is the whole point of the sizing
discussion, and it is a gift of the domain rather than a design flourish.

Six partitions on the events topic, one on the faults topic. Ask yourself why the
number is driven by consumer parallelism and not by bandwidth; we come back to it in
the last section of the other notebook.

In [3]:
topics = [NewTopic(TOPIC_EVENTS, num_partitions=6, replication_factor=1),
          NewTopic(TOPIC_FAULTS, num_partitions=1, replication_factor=1)]

for t, f in admin.create_topics(topics).items():
    try:
        f.result()
        print("created", t)
    except Exception as e:
        print("topic", t, "->", e)

created elevator-events
created elevator-faults


In [4]:
md_ = admin.list_topics(timeout=10)
for t in md_.topics.values():
    if t.topic.startswith("elevator"):
        print(f'"{t.topic}" with {len(t.partitions)} partition(s)')

"elevator-faults" with 1 partition(s)
"elevator-events" with 6 partition(s)


## The fleet

24 units in three regions, eight each. Small enough to run on a laptop, and arranged
so that **each region peaks at a different minute of the same absolute timeline** —
because rush hour is local. That single fact is the strongest argument in the lecture
for partitioning by region rather than by unit, and the other notebook measures it.

Event time lives in the payload, so this producer can send everything as fast as it
likes: nothing here depends on how quickly you run the cells.

In [5]:
# -*- coding: utf-8 -*-
"""The elevator fleet trace — one source for the simulator notebook and for the
verification run.

Designed, not random. The background traffic is there so that the fleet
aggregation and the sizing arithmetic have something plausible to chew on; every
other section exists to isolate exactly one parameter, which is the same
discipline the EPL modules use for their traces.

Event time lives in the payload, so the producer can send everything as fast as
it likes: nothing here depends on wall-clock speed.
"""
from datetime import datetime, timedelta

T0 = datetime(2026, 10, 15, 8, 0, 0)          # fleet time zero, UTC

REGIONS = {'EU': 'Europe/Milan', 'US': 'America/New_York', 'JP': 'Asia/Tokyo'}
UNITS = [f'{r}-{i:03d}' for r in REGIONS for i in range(1, 9)]   # 24 units, 8 per region

# Local rush hour is local: on ONE absolute timeline each region peaks at a
# different minute. Rate = doors per unit per minute, indexed by minute 0..5.
RUSH = {
    'EU': [1, 2, 6, 6, 2, 1],     # Milan is in it
    'US': [1, 1, 1, 2, 4, 6],     # New York is waking up
    'JP': [6, 4, 2, 1, 1, 1],     # Tokyo is past it
}

FLOORS = 10


def ev(kind, unit, ts, **kw):
    d = {'event': kind, 'unitId': unit, 'ts': ts.isoformat(timespec='seconds')}
    d.update(kw)
    return d


def background():
    """Six minutes of fleet time, 24 units, one absolute clock, three peaks."""
    out = []
    for unit in UNITS:
        region = unit.split('-')[0]
        seed = int(unit.split('-')[1])
        for minute, rate in enumerate(RUSH[region]):
            for k in range(rate):
                # spread inside the minute, deterministically
                sec = (seed * 7 + k * 17) % 60
                ts = T0 + timedelta(minutes=minute, seconds=sec)
                car = 'A' if (seed + k) % 2 == 0 else 'B'
                floor = (seed * 3 + k * 5) % FLOORS
                out.append(ev('DoorOpened', unit, ts, car=car, floor=floor,
                              servedDir='UP' if (seed + k) % 2 == 0 else 'DOWN'))
    return sorted(out, key=lambda e: e['ts'])


def bunching():
    """Three configurations at one unit, each isolating one parameter of the
    bunching rule `same floor, same servedDir, different car, within 20 s`."""
    u, base = 'EU-003', T0 + timedelta(minutes=7)
    return [
        # (a) a real bunching: 8 s apart, same floor, same direction, two cars
        ev('DoorOpened', u, base,                        car='A', floor=3, servedDir='UP'),
        ev('DoorOpened', u, base + timedelta(seconds=8), car='B', floor=3, servedDir='UP'),
        # (b) too far apart: 25 s, everything else equal -> the guard must cut it
        ev('DoorOpened', u, base + timedelta(seconds=60), car='A', floor=5, servedDir='DOWN'),
        ev('DoorOpened', u, base + timedelta(seconds=85), car='B', floor=5, servedDir='DOWN'),
        # (c) close in time, same floor, but the other direction -> not a bunching
        ev('DoorOpened', u, base + timedelta(seconds=120), car='A', floor=7, servedDir='UP'),
        ev('DoorOpened', u, base + timedelta(seconds=126), car='B', floor=7, servedDir='DOWN'),
    ]


def straggler():
    """One unit drops off the network and comes back with a backlog.

    JP-005 stops reporting after minute 5 and reconnects at minute 9, replaying
    six doors whose event time is minutes 6 and 7 — six minutes behind the rest
    of the fleet by the time they arrive."""
    u = 'JP-005'
    out = []
    for k in range(6):
        ts = T0 + timedelta(minutes=6 + k // 3, seconds=10 * (k % 3))
        out.append(ev('DoorOpened', u, ts, car='A' if k % 2 else 'B',
                      floor=(k * 3) % FLOORS, servedDir='UP'))
    return out       # produced LAST, long after the rest


def faults():
    """Rare, high value, and on their own topic."""
    return [
        ev('Fault', 'EU-003', T0 + timedelta(minutes=2, seconds=30), code='DOOR_OBSTRUCTED'),
        ev('Fault', 'US-006', T0 + timedelta(minutes=4, seconds=5),  code='OVERSPEED'),
        ev('Fault', 'JP-005', T0 + timedelta(minutes=5, seconds=50), code='COMMS_LOST'),
        ev('Fault', 'EU-001', T0 + timedelta(minutes=8, seconds=12), code='DOOR_OBSTRUCTED'),
    ]


def reference():
    """The static side of the stream-static join: unit -> building, region."""
    return [{'unitId': u,
             'buildingId': f'B{u.split("-")[1]}',
             'region': REGIONS[u.split('-')[0]]}
            for u in UNITS]


SECTIONS = [('1 background', background),
            ('2 bunching', bunching),
            ('3 straggler', straggler)]

In [6]:
for name, fn in SECTIONS:
    rows = fn()
    print(f"{name:14s} {len(rows):4d} events   {rows[0]['ts']} .. {rows[-1]['ts']}")
print(f"{'4 faults':14s} {len(faults()):4d} events")
print(f"units {len(UNITS)}, regions {len(REGIONS)}")

1 background    384 events   2026-10-15T08:00:00 .. 2026-10-15T08:05:59
2 bunching        6 events   2026-10-15T08:07:00 .. 2026-10-15T08:09:06
3 straggler       6 events   2026-10-15T08:06:00 .. 2026-10-15T08:07:20
4 faults          4 events
units 24, regions 3


In [7]:
producer = SerializingProducer({
    "bootstrap.servers": bootstrap_server,
    "key.serializer": StringSerializer("utf_8"),
    "value.serializer": StringSerializer("utf_8"),
})

def send(rows, topic=None):
    """key = unitId, deliberately: it is the partitioning key of the lecture."""
    for r in rows:
        producer.produce(topic=topic or TOPIC_EVENTS,
                         key=r["unitId"], value=json.dumps(r))
    producer.flush()
    print(f"sent {len(rows)} events to {topic or TOPIC_EVENTS}")

---
## 1) Background traffic — six minutes of fleet time

Sent one minute at a time, so you can watch the aggregations move in the other
notebook. Run the cell six times, or all at once: the result is the same, because
the timestamps are in the payload.

In [8]:
bg = background()
for m in range(6):
    send([e for e in bg if e["ts"][14:16] == f"{m:02d}"])
    time.sleep(1)

sent 64 events to elevator-events
sent 56 events to elevator-events
sent 72 events to elevator-events
sent 72 events to elevator-events
sent 56 events to elevator-events
sent 64 events to elevator-events


In [9]:
send(faults(), TOPIC_FAULTS)

sent 4 events to elevator-faults


---
## 2) One bunching, and two things that are not

Two cars opening at the same floor, in the same direction, within twenty seconds is
**bunching** — two cars serving one call, so the next call waits. The section sends
three configurations at unit `EU-003`, and only the first is a bunching:

| | events | what it isolates |
|---|---|---|
| a | floor 3, both UP, cars A and B, **8 s** apart | the real thing |
| b | floor 5, both DOWN, two cars, **25 s** apart | too far apart in time |
| c | floor 7, **8 s** apart, but UP then DOWN | same floor, opposite directions |

If the query returns anything other than exactly one row, it is wrong.

In [10]:
send(bunching())

sent 6 events to elevator-events


---
## 3) A unit that reconnects with a backlog

`JP-005` drops off the network after minute 5 and comes back at minute 9, replaying
six door events whose **event time** is minutes 6 and 7. By the time they arrive the
rest of the fleet has moved on.

This is the long tail of any real fleet — a lift in a basement, a router rebooting —
and it is the case that breaks watermarks. Predict, before running the cell: **how
many of the six will be counted?**

In [11]:
send(straggler())

sent 6 events to elevator-events


---
## Clean up

Delete the two topics, so the next run of this module starts from nothing.

In [12]:
for t, f in admin.delete_topics([TOPIC_EVENTS, TOPIC_FAULTS],
                                operation_timeout=30).items():
    try:
        f.result()
        print("deleted", t)
    except Exception as e:
        print("failed to delete", t, e)

deleted elevator-events
deleted elevator-faults
